In [62]:
from importlib.metadata import version

# Load the file we'll tokenize

* It's a public domain short story, [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict)

In [63]:
# Ensure the file exists locally

import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

print("The book should now exist on your local disk!")

The book should now exist on your local disk!


In [64]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters: ", len(raw_text))
print(raw_text[:99])

Total number of characters:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## Version 1 - Very simple, regex based tokenizer

### Step 1 - Tokenize the text

In [65]:
# Simply split based on spaces. Use a toy-example for now

import re

text="Hello, reader. This, is a test of tokenization."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'reader.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test', ' ', 'of', ' ', 'tokenization.']


In [66]:
# Also split based on commas, periods

result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'reader', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', ' ', 'of', ' ', 'tokenization', '.', '']


In [67]:
# Remove the empty strings

result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'reader', '.', 'This', ',', 'is', 'a', 'test', 'of', 'tokenization', '.']


In [68]:
# Handle even more types of punctuations

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'reader', '.', 'This', ',', 'is', 'a', 'test', 'of', 'tokenization', '.']


In [69]:
# Split the actual book text

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [70]:
# Print the number of tokens

print(len(preprocessed))

4690


### Step 2: Convert tokens into token IDs

In [71]:
# Build a vocabulary using all the unique tokens

all_tokens = sorted(set(preprocessed))
vocab_size = len(all_tokens)
print(vocab_size)

1130


In [72]:
vocab = {token:integer for integer, token in enumerate(all_tokens)}

for i, item in list(enumerate(vocab.items()))[:75]:
    print(item)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)
('How', 52)
('I', 53)
('If', 54)
('In', 55)
('It', 56)
('Jack', 57)
('Jove', 58)
('Just', 59)
('Lord', 60)
('Made', 61)
('Miss', 62)
('Money', 63)
('Monte', 64)
('Moon-dancers', 65)
('Mr', 66)
('Mrs', 67)
('My', 68)
('Never', 69)
('No', 70)
('Now', 71)
('Nutley', 72)
('Of', 73)
('Oh', 74)


In [73]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.vocab = vocab # Maps strings to integers
        self.vocab_reverse = {i:s for s,i in vocab.items()}

    def encode(self, text):
        # Split into tokens
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [token for token in tokens if token.strip()]

        # Find the corresponding token IDs
        ids = [self.vocab[token] for token in tokens]
        return ids
        
    def decode(self, ids):
        # Find the corresponding strings
        tokens = [self.vocab_reverse[id] for id in ids]

        # Concatenate them
        text = " ".join(tokens)

        # Remove spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [74]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [75]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [76]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

### Step 3: Handle unknown tokens and multiple texts

In [77]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [78]:
len(vocab.items())

1132

In [79]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [80]:
# Update the tokenizer to use these new tokens


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.vocab = vocab
        self.vocab_reverse = {integer: string for string, integer in vocab.items()}

    def encode(self, text):
        # Split into tokens
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [token for token in tokens if token.strip()]

        # Replace unknown tokens appropriately
        tokens = [token if token in self.vocab else "<|unk|>" for token in tokens]

        # Find the corresponding token IDs
        ids = [self.vocab[token] for token in tokens]
        return ids

    def decode(self, ids):
        # Find the corresponding strings
        tokens = [self.vocab_reverse[id] for id in ids]

        # Concatenate them
        text = " ".join(tokens)

        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r"\1", text)

        return text

In [81]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [82]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [83]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

### Step 4 - Use Byte Pair Encoder via Tiktoken

If we tokenize based on characters, then the vocab is very compact (say 26 chars for English), but we don't encode much meaning.

If we tokenize based on whole words, and even use the "unk" token, then we don't really capture a lot of meaning for all the unseen words (they all get the same token id).

1. Encodes some meaning for all text (robust to rare or unseen words)

* BPE breaks words into subword units.
* Allows GPT to process:
    * rare words
    * misspellings
    * made-up words
    * long compounds
    * multilingual and code text
* No “unknown token” problem.

2. Keeps the vocab size within check

* Word-level vocabularies would require millions of entries.
* BPE keeps vocab sizes manageable (≈50k tokens).
* Smaller vocab → faster training, smaller embedding matrices, less memory use.

3. Compresses text effectively

* Frequent character pairs merge into longer tokens.
* Common words and morphemes become single tokens.
* Fewer tokens per sentence → faster inference + longer context windows.

4. Captures linguistic structure naturally

* Learned merges produce meaningful subwords:
    * prefixes (e.g., un-, re-)
    * stems (e.g., struct, form)
    * suffixes (e.g., -ing, -tion)
* Helps the model learn relationships among word parts.

5. Deterministic, reversible, and stable

* Same text + same merge rules → same tokenization every time.
* Tokens can be decoded back to the exact original string.
* Reliable for training and inference.

In [84]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [85]:
tokenizer = tiktoken.get_encoding("gpt2")

In [86]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [87]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


### Step 5 - Prepare Input - Output Pairs for Training using a Sliding Window Approach

To train LLMs to generate one word at a time, we want to prepare the training data accordingly where the next word in a sequence represents the target to predict.

In [91]:
import torch
from torch.utils.data import Dataset


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, context_size, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Chunk the text into overlapping sequences of size `context_size`
        for i in range(0, len(token_ids) - context_size, stride):
            input_chunk = token_ids[i : i + context_size]
            output_chunk = token_ids[i + 1 : i + context_size + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [92]:
from torch.utils.data import DataLoader


def create_dataloader_v1(
    txt,
    batch_size=4,
    context_size=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, context_size, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )

    return dataloader

In [93]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, context_size=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [94]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [95]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, context_size=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Step 6 - Create Semantic (Token) Embeddings

Usually, these embedding layers are part of the LLM itself 
and are updated (trained) during model training

![Token Embedding Image](images/token_embedding.png)

In [112]:
# Decide on the dimensions for the token embedding layer

vocab_size = tiktoken.get_encoding("gpt2").n_vocab
print(vocab_size)

token_embedding_output_dim = 256

50257


In [113]:
# Create the token embedding layer

token_embedding_layer = torch.nn.Embedding(vocab_size, token_embedding_output_dim)

In [114]:
# Pass the input from the first batch (created above) through this layer
token_embedding = token_embedding_layer(inputs)
print(token_embedding.shape)
# 8 inputs in each batch, 4 tokens in each input, 256 dimensions for each token

print(token_embedding)

torch.Size([8, 4, 256])
tensor([[[-2.1359,  0.9037,  2.3032,  ..., -1.1645, -0.3450,  0.1219],
         [ 0.9627, -1.7072, -0.7038,  ...,  0.7870, -0.5234, -0.4941],
         [ 0.6509,  0.3263,  0.9172,  ..., -0.5481,  1.6323,  0.5851],
         [ 0.0207, -2.2827,  0.1278,  ..., -0.8284, -1.7556, -0.5384]],

        [[-0.1825,  0.8046, -1.3454,  ...,  1.2595, -0.5655,  1.7161],
         [-0.6925,  2.6274, -0.3019,  ...,  0.3556,  0.0531, -0.5258],
         [-1.6254, -0.5082,  0.4007,  ...,  0.6828, -0.3807,  0.4285],
         [ 0.6032, -0.9518,  0.2168,  ..., -1.4074, -0.7987, -0.7034]],

        [[-0.0054, -0.5251,  0.3163,  ..., -1.3411, -1.2116,  0.8989],
         [ 1.9007,  0.8929, -1.4715,  ..., -0.0223, -0.9270,  0.5536],
         [ 1.1506,  1.0163,  1.4216,  ...,  0.2184,  0.2405,  0.1082],
         [ 1.0240,  0.9632,  1.0466,  ...,  1.4006, -0.8571, -0.5560]],

        ...,

        [[-1.3064, -0.5001,  0.9673,  ...,  0.6214, -1.5296, -1.6237],
         [ 0.6656,  0.6963, -1.91

### Step 7 - Create Positional Embeddings

Since the above token embeddings don't capture any positional information, let's also do that.

Since the tokens can only be at positions 0 to position context_size - 1, our layer's first dimension will be context_size, second one will be whatever we wanna pick (ex: 256)

In [ ]:
# Decide on the dimensions for the positional embedding layer

position_embedding_number_of_positions = 4
positional_embedding_output_dim = 256

In [ ]:
# Create the positional embedding layer

positional_embedding_layer = torch.nn.Embedding(
    position_embedding_number_of_positions, positional_embedding_output_dim
)